In [ ]:
%load_ext autoreload
%autoreload 2

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "3" 
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["VECLIB_MAXIMUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"
os.environ["POLARS_MAX_THREADS"] = "8" # Polars можно дать чуть больше

import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Current GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

CUDA available: True
Current GPU: NVIDIA RTX A4000


In [ ]:
import gc
import time
import numpy as np
import polars as pl
import kagglehub
from kagglehub import KaggleDatasetAdapter
from catboost import CatBoostClassifier, CatBoostRanker, Pool

In [ ]:
dataset_path = "thekabeton/ysda-recsys-2026-yambda-dataset/versions/3"
data_orig = kagglehub.dataset_load(KaggleDatasetAdapter.POLARS, dataset_path, "likes.parquet").collect()
artists = kagglehub.dataset_load(KaggleDatasetAdapter.POLARS, dataset_path, "artist_item_mapping_small.parquet").collect()
artists = artists.with_columns([
    pl.col("item_id").cast(pl.UInt32),
    pl.col("artist_id").cast(pl.UInt32)
])

In [ ]:
ENABLE_LOG = 1
def log(args):
    if ENABLE_LOG:
        print(args)

def recall_at_k(y_true_df, y_pred_df, k=100):
    top_k_pred = (
        y_pred_df.sort(["uid", "score"], descending=[False, True])
        .group_by("uid")
        .head(k)
    )
    hits = top_k_pred.join(y_true_df, on=["uid", "item_id"], how="inner")
    return hits.height / y_true_df.height

In [ ]:
def get_latest_events(df: pl.DataFrame, n: int = 5000) -> pl.DataFrame:
    """
    Возвращает N самых последних событий на основе колонки timestamp.
    """
    return (
        df
        .sort("timestamp", descending=False) # Сортируем от старых к новым
        .tail(n)                             # Берем N последних строк (самые новые)
    )

def smart_sampling(df, max_events_per_user=3, target_rows=None):
    log(f"Исходный размер: {df.height} строк")

    # Сортируем по времени (новые сверху), чтобы head(N) взял последние события
    df_sorted = df.sort(["uid", "timestamp"], descending=[False, True])

    # Группируем по юзеру и берем топ N записей
    df_sampled = df_sorted.group_by("uid").head(max_events_per_user)
    
    log(f"Размер после ограничения истории ({max_events_per_user} на юзера): {df_sampled.height} строк")

    if target_rows and df_sampled.height > target_rows:
        fraction = target_rows / df_sampled.height
        unique_users = df_sampled.select("uid").unique()
        sampled_users = unique_users.sample(fraction=fraction, seed=42)
        df_sampled = df_sampled.join(sampled_users, on="uid", how="inner")
        log(f"Размер после жесткого ограничения: {df_sampled.height} строк")

    return df_sampled.sort("timestamp")

# data_orig_last5k = get_latest_events(data_orig, 5000)

data = smart_sampling(data_orig, max_events_per_user=5)
gc.collect()


data = data.sort('timestamp')
n = len(data)
split_1 = int(n * 0.60)
split_2 = int(n * 0.80)

print(f"Total rows: {n}")

data_hist = data.head(split_1)
data_train = data.slice(split_1, split_2 - split_1)
data_val = data.tail(n - split_2)

print(f"History: {len(data_hist)}, Train: {len(data_train)}, Val: {len(data_val)}")

Исходный размер: 89334605 строк
Размер после ограничения истории (5 на юзера): 3739556 строк
Total rows: 3739556
History: 2243733, Train: 747911, Val: 747912


In [ ]:
from cg.candgen import (
    RetrievalStage,
    
    IALSItemToItemGenerator,
    IALSCandidateGenerator, 
    GlobalPopularityGenerator, 
    ArtistPopularityGenerator, 
    CoVisitationGenerator,   # Новый
    EASEGenerator            # Новый
)

from features.feature_manager import (
    FeatureManager,

    ItemStaticFeatureSource, 
    UserStaticFeatureSource, 
    IALSDotProductSource, 
    CandidateSourceFeatureSource,
    ItemTrendFeatureSource,              # НОВОЕ
    UserArtistAffinityFeatureSource,     # НОВОЕ
    ArtistStaticFeatureSource            # НОВОЕ
)
from main import RecSysPipeline, calc_stats

## Берем маленький датасет для тестов

In [ ]:
data = data_orig.clone()

def get_sized_data(data, size=1):
    if size == 1:
        l = 0.93
        r = 0.98

        als_train = 'quick_val_train'
        als_val = 'quick_val_hist'
    elif size == 2:
        l = 0.75
        r = 0.9

        als_train = 'good_val_train'
        als_val = 'good_val_hist'
    elif size == 3:
        raise 'HAHAHA EBLAN 3'
    else:
        raise 'HAHAHA EBLAN'

    min_t = data["timestamp"].min()
    max_t = data["timestamp"].max()
    duration = max_t - min_t

    t1 = min_t + int(duration * l)
    t2 = min_t + int(duration * r)

    # Разрезаем
    data_hist_train = data.filter(pl.col("timestamp") < t1)
    data_train = data.filter((pl.col("timestamp") >= t1) & (pl.col("timestamp") < t2))

    data_val_hist = data.filter(pl.col("timestamp") < t2)  # <= 0.98
    data_val = data.filter(pl.col("timestamp") >= t2)


    print('История для трейна - data_hist_train: ', data_hist_train.height)
    print('Данные для трейна - data_train: ', data_train.height)

    print('История для трейна валидации - data_val_hist: ', data_val_hist.height)
    print('данные для валидации - data_val: ', data_val.height)

    return data_hist_train, data_train, data_val_hist, data_val, als_train, als_val

data_hist_train, data_train, data_val_hist, data_val, als_train, als_val = get_sized_data(data, 1)

История для трейна - data_hist_train:  80304087
Данные для трейна - data_train:  6505221
История для трейна валидации - data_val_hist:  86809308
данные для валидации - data_val:  2525297


# Эксперементы

In [ ]:
# Кандгены
ials_gen = IALSCandidateGenerator(factors=128, iterations=30, file_prefix=als_train)
ials_i2i = IALSItemToItemGenerator(file_prefix=als_train)
global_gen = GlobalPopularityGenerator(top_n=100)
artist_gen = ArtistPopularityGenerator(artist_mapping=artists)

retrieval = RetrievalStage(generators=[
    # ials_gen, 
    # global_gen, 
    # artist_gen, 
    ials_i2i])

# Фичи и признаки
item_stats = ItemStaticFeatureSource(artist_mapping=artists)
user_stats = UserStaticFeatureSource()
source_stats = CandidateSourceFeatureSource()

feature_manager = FeatureManager(sources=[
    # item_stats, 
    # user_stats, 
    source_stats])

# Собираем пайплайн
pipeline = RecSysPipeline(retrieval_stage=retrieval, feature_manager=feature_manager)

# Обучение пайплайна
pipeline.fit_all(train_history_data=data_hist_train.lazy())

print("\n=== PHASE 1.2: GENERATING TRAIN DATASET ===")
train_lazy = data_train.lazy()
train_uids = data_train["uid"].unique().to_list()

# Создание датасета для обучения
train_dataset = pipeline.create_dataset(
    target_uids=train_uids,
    history_data=data_hist_train.lazy(),
    labels_data=train_lazy,
    num_candidates=300,
    batch_size=100000
)

print(f"Train dataset shape: {train_dataset.shape}")
print(f"Target distribution: \n{train_dataset['target'].value_counts()}")

# Сохранение
# train_dataset.write_parquet(
#     "train_dataset_2-1434.parquet", 
#     compression="snappy", # Быстрое сжатие
#     use_pyarrow=True      # Если установлен pyarrow, работает еще стабильнее
# )

train_dataset

=== Fitting Retrieval Stage ===
=== Fitting Feature Manager ===
      Fit now: candidate_source_stats
=== Fit Complete ===

=== PHASE 1.2: GENERATING TRAIN DATASET ===
Starting dataset generation for 530869 users...
[ials_i2i] Starting up i2i engine...
  Processing batch 0 to 100000...
    Retrieval (Отбор кандидатов)
      Generate now: ials_i2i
    Feature Engineering (Навешивание фичей)
      Extract now: candidate_source_stats
  Processing batch 100000 to 200000...
    Retrieval (Отбор кандидатов)
      Generate now: ials_i2i
    Feature Engineering (Навешивание фичей)
      Extract now: candidate_source_stats
  Processing batch 200000 to 300000...
    Retrieval (Отбор кандидатов)
      Generate now: ials_i2i
    Feature Engineering (Навешивание фичей)
      Extract now: candidate_source_stats
  Processing batch 300000 to 400000...
    Retrieval (Отбор кандидатов)
      Generate now: ials_i2i
    Feature Engineering (Навешивание фичей)
      Extract now: candidate_source_stats
  Pr

uid,item_id,score,is_ials_i2i,source_count,target
u32,u32,f32,f64,f64,i8
29311,8907903,1.003037,1.0,1.0,0
13728,8534867,3.97518,1.0,1.0,0
79522,5235766,0.268089,1.0,1.0,0
87749,7881545,0.878062,1.0,1.0,0
101212,2856436,0.428074,1.0,1.0,0
…,…,…,…,…,…
948613,7582363,4.443565,1.0,1.0,0
999677,8775045,0.769413,1.0,1.0,0
954576,1792458,1.055451,1.0,1.0,0


In [ ]:
calc_stats(train_dataset, data_train)

=== СТАТИСТИКА ПОКРЫТИЯ (RECALL) ===
Всего реальных лайков в периоде: 6046934

Общий Recall системы (все генераторы вместе): 6.23%
Потеряно лайков (не попали в кандидаты): 5670009
  is_ials_i2i Recall: 6.23%

=== 3. КОНВЕРСИЯ ВНУТРИ КАНДИДАТОВ (PRECISION) ===
shape: (1, 5)
┌─────────────┬────────────────┬────────────────┬─────────────┬──────────────────────────┐
│ is_ials_i2i ┆ total_in_group ┆ likes_in_group ┆ %_конверсия ┆ %_contribution_to_recall │
│ ---         ┆ ---            ┆ ---            ┆ ---         ┆ ---                      │
│ f64         ┆ u32            ┆ i64            ┆ f64         ┆ f64                      │
╞═════════════╪════════════════╪════════════════╪═════════════╪══════════════════════════╡
│ 1.0         ┆ 55806798       ┆ 376925         ┆ 0.6754      ┆ 6.23                     │
└─────────────┴────────────────┴────────────────┴─────────────┴──────────────────────────┘


## pipeline для обучения

In [ ]:
# Кандгены
ials_gen = IALSCandidateGenerator(factors=128, iterations=30, file_prefix=als_train)
global_gen = GlobalPopularityGenerator(top_n=100)
artist_gen = ArtistPopularityGenerator(artist_mapping=artists)
ials_i2i = IALSItemToItemGenerator(file_prefix=als_train)

covisitation_gen = CoVisitationGenerator(top_k_similar=50, history_depth=15, file_prefix=als_train)
# Ограничиваем EASE 30 тысячами треков. Если у тебя 64 ГБ ОЗУ, можешь поставить 50000.
ease_gen = EASEGenerator(top_k_items=30000, l2_reg=500.0, file_prefix=als_train) 

retrieval = RetrievalStage(generators=[
    ials_gen, 
    # covisitation_gen, # Обязательно добавь! 
    ease_gen,         # Обязательно добавь!
    global_gen, 
    artist_gen,
    ials_i2i
])

# Считаем фичи
item_stats = ItemStaticFeatureSource(artist_mapping=artists)
user_stats = UserStaticFeatureSource()
ials_dot = IALSDotProductSource(ials_cg=ials_gen)
source_stats = CandidateSourceFeatureSource()

# НОВЫЕ ФИЧИ
item_trend = ItemTrendFeatureSource(days_trend=14)
user_artist_affinity = UserArtistAffinityFeatureSource(artist_mapping=artists)
artist_static = ArtistStaticFeatureSource(artist_mapping=artists)

feature_manager = FeatureManager(sources=[
    artist_static,
    item_stats, 
    user_stats, 
    ials_dot, 
    source_stats,
    item_trend,             # Свежесть трека
    user_artist_affinity   # Сколько лайкал этого артиста (Топ фича!)
])

# Собираем пайплайн
pipeline = RecSysPipeline(retrieval_stage=retrieval, feature_manager=feature_manager)

data_hist_train = data_hist_train.with_columns([
    pl.col("uid").cast(pl.UInt32),
    pl.col("item_id").cast(pl.UInt32)
])

data_train = data_train.with_columns([
    pl.col("uid").cast(pl.UInt32),
    pl.col("item_id").cast(pl.UInt32)
])


# Обучение пайплайна
pipeline.fit_all(train_history_data=data_hist_train.lazy())

print("\n=== PHASE 1.2: GENERATING TRAIN DATASET ===")
train_lazy = data_train.lazy()
train_uids = data_train["uid"].unique().to_list()

# Создание датасета для обучения
train_dataset = pipeline.create_dataset(
    target_uids=train_uids,
    history_data=data_hist_train.lazy(),
    labels_data=train_lazy,
    num_candidates=200,
    batch_size=100000
)

print(f"Train dataset shape: {train_dataset.shape}")
print(f"Target distribution: \n{train_dataset['target'].value_counts()}")

# Сохранение
train_dataset.write_parquet(
    "train_dataset_2-1434.parquet", 
    compression="snappy", # Быстрое сжатие
    use_pyarrow=True      # Если установлен pyarrow, работает еще стабильнее
)

train_dataset

=== Fitting Retrieval Stage ===
[ials] УЖЕ ОБУЧЕН! Нашел кэш в fittingdata/cg/quick_val_train_ials. Пропускаю fit().
[ease] УЖЕ ОБУЧЕН! Пропускаю fit().
[global_pop] Calculating global top...
[global_pop] Saved to fittingdata/cg/global_pop
[artist_pop] Precalculating artist top tracks...
[artist_pop] Fit complete.
=== Fitting Feature Manager ===
      Fit now: artist_static
[artist_static] Fitting artist stats...
      Fit now: item_static
[item_static] Fitting...
      Fit now: user_static
[user_static] Fitting...
      Fit now: ials_dot_product
      Fit now: candidate_source_stats
      Fit now: item_trend
[item_trend] Fitting trend stats...
      Fit now: user_artist_affinity
=== Fit Complete ===

=== PHASE 1.2: GENERATING TRAIN DATASET ===
Starting dataset generation for 530869 users...
[ials] Loading from disk (Stable version)...
[ease] Loading weights to RAM...
[ials_i2i] Starting up i2i engine...
  Processing batch 0 to 100000...
    Retrieval (Отбор кандидатов)
      Generate 

/home/gavrishokni/rec_sys/ContestEngine/features/feature_manager.py:306: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  is_cols = [c for c in candidates.columns if c.startswith("is_")]


  Processing batch 100000 to 200000...
    Retrieval (Отбор кандидатов)
      Generate now: ials
      Generate now: ease
      Generate now: global_pop
      Generate now: artist_pop
      Generate now: ials_i2i
    Feature Engineering (Навешивание фичей)
      Extract now: artist_static
      Extract now: item_static
      Extract now: user_static
      Extract now: ials_dot_product
      Extract now: candidate_source_stats
      Extract now: item_trend
      Extract now: user_artist_affinity
  Processing batch 200000 to 300000...
    Retrieval (Отбор кандидатов)
      Generate now: ials
      Generate now: ease
      Generate now: global_pop
      Generate now: artist_pop
      Generate now: ials_i2i
    Feature Engineering (Навешивание фичей)
      Extract now: artist_static
      Extract now: item_static
      Extract now: user_static
      Extract now: ials_dot_product
      Extract now: candidate_source_stats
      Extract now: item_trend
      Extract now: user_artist_affinity


uid,item_id,score,is_ials,is_ease,is_global_pop,is_artist_pop,is_ials_i2i,artist_total_likes,artist_unique_liked_tracks,item_cnt_likes,item_organic_ratio,user_total_likes,user_unique_items,als_dot,als_rank,source_count,item_likes_last_14d,item_trend_ratio,artist_id_right,user_artist_likes_cnt,target
u32,u32,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f64,f64,f64,f64,u32,u32,i8
170309,3688935,0.490111,1.0,0.0,0.0,0.0,0.0,32648.0,87.0,1399.0,0.348106,46.0,45.0,0.490111,159.0,1.0,120.0,0.085776,398681,0,0
27357,7813350,0.51701,1.0,0.0,0.0,0.0,0.0,29534.0,63.0,1626.0,0.49016,428.0,428.0,0.51701,353.0,1.0,161.0,0.099016,125704,0,0
20197,5041060,0.714978,0.0,0.0,0.0,0.0,1.0,71762.0,122.0,848.0,0.73467,263.0,255.0,0.245555,439.0,1.0,66.0,0.07783,637143,3,0
119949,2070722,0.573618,0.0,0.0,0.0,0.0,1.0,17186.0,37.0,1990.0,0.386432,35.0,34.0,0.209675,115.0,1.0,110.0,0.055276,330504,0,0
25241,8420967,0.042835,1.0,0.0,0.0,0.0,0.0,1992.0,137.0,395.0,0.63038,1.0,1.0,0.042835,116.0,1.0,30.0,0.075949,83901,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
943477,5499247,0.200013,0.0,1.0,0.0,0.0,0.0,6817.0,22.0,510.0,0.494118,110.0,107.0,0.075468,456.0,1.0,17.0,0.033333,1084731,0,0
984267,2014100,0.439612,1.0,0.0,0.0,0.0,0.0,3396.0,13.0,425.0,0.489412,98.0,74.0,0.439611,307.0,1.0,22.0,0.051765,535534,0,0
979580,936611,0.776455,1.0,0.0,0.0,0.0,0.0,14247.0,17.0,1366.0,0.475842,266.0,263.0,0.776455,124.0,1.0,59.0,0.043192,107626,1,0


In [ ]:
# Сохранение
train_dataset.write_parquet(
    "train_dataset_1-0150-w-ease.parquet", 
    compression="snappy", # Быстрое сжатие
    use_pyarrow=True      # Если установлен pyarrow, работает еще стабильнее
)


# Статистики по датасету

In [ ]:
calc_stats(train_dataset, data_train)

=== СТАТИСТИКА ПОКРЫТИЯ (RECALL) ===
Всего реальных лайков в периоде: 6505221

Общий Recall системы (все генераторы вместе): 17.00%
Потеряно лайков (не попали в кандидаты): 5399017
  is_ials Recall: 6.81%
  is_ease Recall: 7.32%
  is_global_pop Recall: 2.54%
  is_artist_pop Recall: 2.62%
  is_ials_i2i Recall: 7.59%

=== 3. КОНВЕРСИЯ ВНУТРИ КАНДИДАТОВ (PRECISION) ===
shape: (31, 9)
┌─────────┬─────────┬────────────┬────────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ is_ials ┆ is_ease ┆ is_global_ ┆ is_artist_ ┆ … ┆ total_in_g ┆ likes_in_ ┆ %_конверс ┆ %_contrib │
│ ---     ┆ ---     ┆ pop        ┆ pop        ┆   ┆ roup       ┆ group     ┆ ия        ┆ ution_to_ │
│ f64     ┆ f64     ┆ ---        ┆ ---        ┆   ┆ ---        ┆ ---       ┆ ---       ┆ recall    │
│         ┆         ┆ f64        ┆ f64        ┆   ┆ u32        ┆ i64       ┆ f64       ┆ ---       │
│         ┆         ┆            ┆            ┆   ┆            ┆           ┆           ┆ f64       │
╞═════════

# Обучение

In [ ]:
стоп

In [ ]:
def prepare_catboost_data(df: pl.DataFrame, is_train: bool = True, neg_fraction: float = 0.05):
    """
    Универсальная функция для Train, Val и Submit.
    is_train=True : Оставляет все лайки, режет нули до neg_fraction и сортирует.
    is_train=False: Ничего не режет, только сортирует и готовит массивы.
    """
    print(f"\n--- Preparing Data (is_train={is_train}) ---")
    print(f"Initial shape: {df.shape}")

    # 1. Сэмплирование (ТОЛЬКО ДЛЯ TRAIN)
    if is_train and "target" in df.columns:
        print("Downsampling negatives...")
        pos_df = df.filter(pl.col("target") == 1)
        neg_df = df.filter(pl.col("target") == 0).sample(fraction=neg_fraction, seed=42)
        df = pl.concat([pos_df, neg_df])
        print(f"Shape after downsampling: {df.shape} ({pos_df.height} positives)")

    # 2. Обязательная сортировка по uid (ТРЕБОВАНИЕ CATBOOST RANKER!)
    df = df.sort("uid")

    # 3. Отбор фичей
    drop_cols = ["uid", "item_id", "target", "artist_id", "source"]
    feature_names = [col for col in df.columns if col not in drop_cols]
    print(f"Features ({len(feature_names)}): {feature_names}")

    # 4. Извлекаем данные (Сжатие до float32 для RAM)
    X = df.select(feature_names).to_numpy().astype(np.float32)
    group_ids = df["uid"].to_numpy().astype(np.uint32)
    
    # Чтобы не потерять item_ids для сабмита, можем их тоже вернуть
    item_ids = df["item_id"].to_numpy().astype(np.uint32)

    # 5. Извлекаем таргет (если есть)
    y = None
    if "target" in df.columns:
        y = df["target"].to_numpy().astype(np.int8)

    return X, y, group_ids, item_ids, feature_names

In [ ]:
train_dataset = pl.read_parquet('train_dataset_1-0150-w-ease.parquet')

In [ ]:
calc_stats(train_dataset, data_train)

=== СТАТИСТИКА ПОКРЫТИЯ (RECALL) ===
Всего реальных лайков в периоде: 16854532

Общий Recall системы (все генераторы вместе): 9.57%
Потеряно лайков (не попали в кандидаты): 15241485
  is_ials Recall: 5.41%
  is_global_pop Recall: 3.24%
  is_artist_pop Recall: 2.19%

=== 3. КОНВЕРСИЯ ВНУТРИ КАНДИДАТОВ (PRECISION) ===
shape: (7, 7)
┌─────────┬──────────────┬──────────────┬──────────────┬──────────────┬─────────────┬──────────────┐
│ is_ials ┆ is_global_po ┆ is_artist_po ┆ total_in_gro ┆ likes_in_gro ┆ %_конверсия ┆ %_contributi │
│ ---     ┆ p            ┆ p            ┆ up           ┆ up           ┆ ---         ┆ on_to_recall │
│ f64     ┆ ---          ┆ ---          ┆ ---          ┆ ---          ┆ f64         ┆ ---          │
│         ┆ f64          ┆ f64          ┆ u32          ┆ i64          ┆             ┆ f64          │
╞═════════╪══════════════╪══════════════╪══════════════╪══════════════╪═════════════╪══════════════╡
│ 1.0     ┆ 1.0          ┆ 1.0          ┆ 248880       ┆ 8191 

In [ ]:
# 1. Готовим данные для обучения (оставляем 5% негативов)
X_train, y_train, group_train, _, feat_names = prepare_catboost_data(
    train_dataset, 
    is_train=True, 
    neg_fraction=0.05
)

# Удаляем исходный датасет из памяти, он больше не нужен!
del train_dataset
gc.collect()

print("Creating Classifier Pool...")
train_pool = Pool(
    data=X_train, 
    label=y_train,
    # group_id здесь НЕ обязателен для Logloss, можно убрать для стабильности
    feature_names=feat_names
)

# Очистим память перед боем
del X_train, y_train
gc.collect()

# 2. Настройка КЛАССИФИКАТОРА
print("Starting CatBoost Classifier Training...")

model = CatBoostClassifier(
    iterations=700,
    learning_rate=0.1,
    depth=6,
    
    # Стандартные параметры
    loss_function='Logloss',
    eval_metric='AUC',       # AUC на GPU считается мгновенно и без ошибок
    
    task_type='GPU',
    gpu_ram_part=0.8,
    random_seed=42,
    
    # Режим "тишины", чтобы Jupyter не захлебнулся
    logging_level='Silent',
    allow_writing_files=False
)

try:
    print("Training in progress... (звездочка [*] - это нормально)")
    model.fit(train_pool)
    print("✅ УСПЕХ! МОДЕЛЬ ОБУЧЕНА.")
    model.save_model("catboost_classifier_stable.cbm")
except Exception as e:
    print(f"❌ Ошибка: {e}")

print(f"Best score (AUC): {model.get_best_score()}")


--- Preparing Data (is_train=True) ---
Initial shape: (179796589, 14)
Downsampling negatives...
Shape after downsampling: (10522224, 14) (1613047 positives)
Features (10): ['score', 'is_ials', 'is_global_pop', 'is_artist_pop', 'item_cnt_likes', 'item_organic_ratio', 'user_total_likes', 'user_unique_items', 'als_dot', 'source_count']
Creating Classifier Pool...
Starting CatBoost Classifier Training...
Training in progress... (звездочка [*] - это нормально)


Default metric period is 5 because AUC is/are not implemented for GPU


✅ УСПЕХ! МОДЕЛЬ ОБУЧЕНА.
Best score (AUC): {'learn': {'Logloss': 0.3665881851593351}}


# Валидация

In [ ]:
# Сохранение
# ranker_model.save_model("catboost_model_last_70.cbm")

# # Загрузка (позже, когда понадобится)
# from catboost import CatBoostClassifier
# loaded_model = CatBoostClassifier()
# loaded_model.load_model("catboost_model.cbm")

In [ ]:
data_hist_train, data_train, data_val_hist, data_val, als_train, als_val

In [ ]:
# Кандгены
ials_gen = IALSCandidateGenerator(factors=128, iterations=30, file_prefix=als_val)
global_gen = GlobalPopularityGenerator(top_n=100)
artist_gen = ArtistPopularityGenerator(artist_mapping=artists)

retrieval = RetrievalStage(generators=[ials_gen, global_gen, artist_gen])

# Фичи и признаки
item_stats = ItemStaticFeatureSource(artist_mapping=artists)
user_stats = UserStaticFeatureSource()
ials_dot = IALSDotProductSource(ials_cg=ials_gen)
source_stats = CandidateSourceFeatureSource()

feature_manager = FeatureManager(sources=[item_stats, user_stats, ials_dot, source_stats])

# Собираем пайплайн
pipeline = RecSysPipeline(retrieval_stage=retrieval, feature_manager=feature_manager)

# Обучение пайплайна
pipeline.fit_all(train_history_data=data_val_hist.lazy())

print("\n=== PHASE 1.2: GENERATING VAL DATASET ===")
val_uids = data_val["uid"].unique().to_list()
val_labels_lazy = data_val.lazy()

# Создание датасета для валидации
val_dataset = pipeline.create_dataset(
    target_uids=val_uids,
    # ИСПОЛЬЗУЕМ ПОЛНУЮ ИСТОРИЮ (A + B)
    history_data=data_val_hist.lazy(), 
    # ТАРГЕТЫ ИЗ ОКНА C
    labels_data=val_labels_lazy,
    num_candidates=200,
    batch_size=50000
)

print(f"Val dataset shape: {val_dataset.height}")
# Проверь, что в target_counts есть единицы!
print(f"Target distribution: \n{val_dataset['target'].value_counts()}")

# Сохранение
val_dataset.write_parquet(
    "val_dataset_2-1434.parquet", 
    compression="snappy", # Быстрое сжатие
    use_pyarrow=True      # Если установлен pyarrow, работает еще стабильнее
)

val_dataset

=== Fitting Retrieval Stage ===
[ials] УЖЕ ОБУЧЕН! Нашел кэш в fittingdata/cg/good_val_hist_ials. Пропускаю fit().
[global_pop] Calculating global top...
[global_pop] Saved to fittingdata/cg/global_pop
[artist_pop] Precalculating artist top tracks...
[artist_pop] Fit complete.
=== Fitting Feature Manager ===
      Fit now: item_static
[item_static] Fitting...
      Fit now: user_static
[user_static] Fitting...
      Fit now: ials_dot_product
      Fit now: candidate_source_stats
=== Fit Complete ===

=== PHASE 1.2: GENERATING VAL DATASET ===
Starting dataset generation for 635729 users...
[ials] Loading from disk (Stable version)...
  Processing batch 0 to 50000...
    Retrieval (Отбор кандидатов)
      Generate now: ials
      Generate now: global_pop
      Generate now: artist_pop
    Feature Engineering (Навешивание фичей)
      Extract now: item_static
      Extract now: user_static
      Extract now: ials_dot_product
      Extract now: candidate_source_stats


/home/gavrishokni/rec_sys/ContestEngine/features/feature_manager.py:187: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  is_cols = [c for c in candidates.columns if c.startswith("is_")]


  Processing batch 50000 to 100000...
    Retrieval (Отбор кандидатов)
      Generate now: ials
      Generate now: global_pop
      Generate now: artist_pop
    Feature Engineering (Навешивание фичей)
      Extract now: item_static
      Extract now: user_static
      Extract now: ials_dot_product
      Extract now: candidate_source_stats
  Processing batch 100000 to 150000...
    Retrieval (Отбор кандидатов)
      Generate now: ials
      Generate now: global_pop
      Generate now: artist_pop
    Feature Engineering (Навешивание фичей)
      Extract now: item_static
      Extract now: user_static
      Extract now: ials_dot_product
      Extract now: candidate_source_stats
  Processing batch 150000 to 200000...
    Retrieval (Отбор кандидатов)
      Generate now: ials
      Generate now: global_pop
      Generate now: artist_pop
    Feature Engineering (Навешивание фичей)
      Extract now: item_static
      Extract now: user_static
      Extract now: ials_dot_product
      Extract 

uid,item_id,score,is_ials,is_global_pop,is_artist_pop,item_cnt_likes,item_organic_ratio,artist_id,user_total_likes,user_unique_items,als_dot,source_count,target
u32,u32,f32,f64,f64,f64,u32,f64,u32,u32,u32,f32,f64,i8
67451,6762454,0.214044,1.0,0.0,0.0,7046,0.468351,1158635,20,18,0.214044,1.0,0
29750,1051073,35139.0,0.0,1.0,0.0,35139,0.50175,1031886,123,108,0.280377,1.0,0
60101,3784097,0.243588,1.0,0.0,0.0,6672,0.526529,792320,116,104,0.243588,1.0,0
3524,6813373,21435.0,0.0,1.0,0.0,21435,0.690506,1079075,1626,1616,0.566243,1.0,0
34346,1432819,25703.0,0.0,1.0,0.0,25703,0.594366,1029533,41,41,0.059816,1.0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…
976815,4892926,23970.0,0.0,1.0,0.0,23970,0.573175,1214210,39,37,0.246923,1.0,0
946388,8743390,35501.0,0.0,1.0,0.0,35501,0.605222,731504,96,95,0.217106,1.0,0
961927,2846168,847.0,0.0,0.0,1.0,847,0.698937,1278569,101,99,0.302777,1.0,0


In [ ]:
print("\n=== STARTING VALIDATION INFERENCE ===")

# 1. Подготовка признаков (is_train=False — НИЧЕГО НЕ УДАЛЯЕМ)
# Нам нужны все 200 кандидатов на юзера, чтобы честно выбрать топ-100
X_val, y_val, group_val, items_val, _ = prepare_catboost_data(
    val_dataset, 
    is_train=False 
)

# 2. Предсказание вероятностей (у Классификатора берем колонку 1)
print("Predicting probabilities on GPU...")
try:
    # Метод predict_proba вернет матрицу [N, 2], нам нужна вероятность лайка (индекс 1)
    val_preds = model.predict_proba(X_val)[:, 1]
    
    # 3. Собираем результаты в Polars DataFrame для подсчета метрики
    y_pred_df = pl.DataFrame({
        "uid": group_val,
        "item_id": items_val,
        "score": val_preds
    })

    print("Success! Results gathered.")
except Exception as e:
    print(f"❌ Ошибка при инференсе: {e}")

# 4. Освобождаем память от тяжелых Numpy массивов
del X_val, val_preds
gc.collect()

# 5. Сравнение с истинными лайками (Ground Truth)
# Нам нужны пары (uid, item_id) из периода валидации data_val
y_true_val_df = data_val.select(["uid", "item_id"])

print(f"Calculating Recall@100 for {y_pred_df['uid'].n_unique()} users...")

# Твоя функция recall_at_k
r100 = recall_at_k(y_true_val_df, y_pred_df, k=100)

print(f"\n" + "="*40)
print(f"🔥 FINAL VALIDATION RESULT 🔥")
print(f"Recall@100: {r100:.6f}")
print("="*40)



=== STARTING VALIDATION INFERENCE ===

--- Preparing Data (is_train=False) ---
Initial shape: (188726870, 14)
Features (10): ['score', 'is_ials', 'is_global_pop', 'is_artist_pop', 'item_cnt_likes', 'item_organic_ratio', 'user_total_likes', 'user_unique_items', 'als_dot', 'source_count']
Predicting probabilities on GPU...
Success! Results gathered.
Calculating Recall@100 for 635729 users...

🔥 FINAL VALIDATION RESULT 🔥
Recall@100: 0.060378


# Submitter

In [ ]:
хуй

In [ ]:
class Submitter:
    def __init__(self, model, pipeline, test_data=None):
        """
        :param model: Обученная модель CatBoostClassifier
        :param pipeline: Настроенный объект RecSysPipeline
        """
        self.model = model
        self.pipeline = pipeline
        self.test_data = test_data

    def create_submission(self, test_users_df: pl.DataFrame, all_data: pl.LazyFrame, output_path: str = 'submission.csv', num_candidates: int = 200, batch_size: int = 10000):
        print("\n" + "="*40)
        print("🚀 STARTING FINAL SUBMISSION GENERATION 🚀")
        print("="*40)

        if self.test_data is None:
            # 1. Переобучаем статистики на ВСЕХ данных (A+B+C)
            # Это обновит счетчики популярности и виральности на текущий момент
            print("\n[1/5] Re-fitting pipeline features on ALL data...")
            self.pipeline.fit_all(train_history_data=all_data)

            # 2. Генерируем кандидатов и фичи для тестовых пользователей
            test_uids = test_users_df["uid"].unique().to_list()
            print(f"\n[2/5] Generating candidates & features for {len(test_uids)} test users...")
            
            # Используем твой батчевый метод из пайплайна
            # labels_data=None, так как на тесте мы не знаем правильных ответов
            test_dataset = self.pipeline.create_dataset(
                target_uids=test_uids,
                history_data=all_data,
                labels_data=None, 
                num_candidates=num_candidates, 
                batch_size=batch_size
            )
        else:
            test_dataset = self.test_data

        # 3. Подготовка данных для CatBoost (аналог prepare_catboost_data)
        print("\n[3/5] Preparing features for CatBoost...")
        
        # Колонки, которые мы НЕ используем в обучении
        drop_cols = ["uid", "item_id", "target", "artist_id", "source"]
        feature_names = [col for col in test_dataset.columns if col not in drop_cols]
        
        # Перевод в Numpy (float32 для экономии RAM)
        X_test = test_dataset.select(feature_names).to_numpy().astype(np.float32)
        uids_test = test_dataset["uid"].to_numpy().astype(np.uint32)
        items_test = test_dataset["item_id"].to_numpy().astype(np.uint32)

        del test_dataset
        gc.collect()

        # 4. Предсказание вероятностей лайка
        print(f"[4/5] Predicting scores with CatBoost on {len(X_test)} pairs...")
        # Используем predict_proba, так как у нас Classifier
        probs = self.model.predict_proba(X_test)[:, 1]

        # Собираем временный DF для ранжирования
        result_df = pl.DataFrame({
            "uid": uids_test,
            "item_id": items_test,
            "score": probs
        })

        del X_test, probs, uids_test, items_test
        gc.collect()

        # 5. Форматирование Топ-100 (Recall@100 требует именно 100 треков)
        print("\n[5/5] Formatting Top-100 recommendations...")
        submission = (
            result_df
            # Сортируем: сначала по юзеру, затем по скору (убывание)
            .sort(["uid", "score"], descending=[False, True])
            # Берем топ-100 для каждого юзера
            .group_by("uid", maintain_order=True)
            .head(100)
            # Склеиваем треки в строку через пробел
            .group_by("uid")
            .agg(
                pl.col("item_id").cast(pl.Utf8).str.join(" ").alias("item_ids")
            )
        )

        # Гарантируем, что ВСЕ пользователи из теста попали в сабмит
        final_submission = (
            test_users_df.select(pl.col("uid").cast(pl.UInt32))
            .join(submission, on="uid", how="left")
            .with_columns(pl.col("item_ids").fill_null("")) # Если нет кандидатов - пустая строка
            .select(["uid", "item_ids"]) # Оставляем только нужные колонки
        )

        # Сохранение на диск
        final_submission.write_csv(output_path)
        print(f"✅ DONE! Submission saved to: {output_path}")

        return final_submission

In [ ]:
# Загружаем тестовых пользователей
file_path = "test_users.csv" # Либо актуальное название файла из Yambda датасета

test_target_users = kagglehub.dataset_load(
  KaggleDatasetAdapter.POLARS,
  dataset_path, # Используем твою переменную dataset_path из начала ноутбука
  file_path
).collect()

print(f"Total test users: {test_target_users.height}")

Total test users: 26021


In [ ]:
model = CatBoostClassifier()
model.load_model("catboost_classifier_stable.cbm")

submitter = Submitter(model=model, pipeline=None)

# data - это твой полный LazyFrame (все 80 млн событий)
my_submission = submitter.create_submission(
    test_users_df=test_target_users,
    all_data=None,
    output_path='0604-1837.csv',
    num_candidates=200,    
    batch_size=10000       
)

print(my_submission.head())


🚀 STARTING FINAL SUBMISSION GENERATION 🚀

[1/5] Re-fitting pipeline features on ALL data...


AttributeError: 'NoneType' object has no attribute 'fit_all'

In [ ]:
# 1. Создаем финальный ALS (на 100% данных)
ials_final = IALSCandidateGenerator(factors=128, iterations=30, file_prefix="all_data")

# 2. Создаем Retrieval Stage
retrieval_final = RetrievalStage(generators=[
    ials_final, 
    GlobalPopularityGenerator(top_n=100), 
    ArtistPopularityGenerator(artist_mapping=artists)
])

# 3. 🔥 ВАЖНО: Пересобираем Feature Manager с НОВЫМ генератором
ials_dot_final = IALSDotProductSource(ials_cg=ials_final) # Привязываем к ials_final!

feature_manager_final = FeatureManager(sources=[
    ItemStaticFeatureSource(artist_mapping=artists),
    UserStaticFeatureSource(),
    ials_dot_final, 
    CandidateSourceFeatureSource()
])

# 4. Собираем финальный пайплайн
pipeline_final = RecSysPipeline(
    retrieval_stage=retrieval_final, 
    feature_manager=feature_manager_final # Используем новый менеджер
)

# 5. Инициализируем Submitter и запускаем
submitter = Submitter(model=model, pipeline=pipeline_final)

# data - это твой полный LazyFrame (все 80 млн событий)
my_submission = submitter.create_submission(
    test_users_df=test_target_users,
    all_data=data.lazy(), 
    output_path='als_128_catboost_v1.csv',
    num_candidates=200,    
    batch_size=10000       
)

print(my_submission.head())


🚀 STARTING FINAL SUBMISSION GENERATION 🚀

[1/5] Re-fitting pipeline features on ALL data...
=== Fitting Retrieval Stage ===
[ials] УЖЕ ОБУЧЕН! Нашел кэш в fittingdata/cg/all_data_ials. Пропускаю fit().
[global_pop] Calculating global top...
[global_pop] Saved to fittingdata/cg/global_pop
[artist_pop] Precalculating artist top tracks...
[artist_pop] Fit complete.
=== Fitting Feature Manager ===
      Fit now: item_static
[item_static] Fitting...
      Fit now: user_static
[user_static] Fitting...
      Fit now: ials_dot_product
      Fit now: candidate_source_stats
=== Fit Complete ===

[2/5] Generating candidates & features for 26021 test users...
Starting dataset generation for 26021 users...
[ials] Loading from disk (Stable version)...
  Processing batch 0 to 10000...
    Retrieval (Отбор кандидатов)
      Generate now: ials
      Generate now: global_pop
      Generate now: artist_pop
    Feature Engineering (Навешивание фичей)
      Extract now: item_static
      Extract now: user